**Q1. Embedding a query**

In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)
v1[0]

np.float64(-0.02058203437252893)

**Q2. Cosine similarity**

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

d = next(
    (
        doc.get("content")
        for doc in documents
        if doc.get("filename") == "02-vector-search/lessons/07-sqlitesearch-vector.md"
    ),
    None,
)

dv = embed.encode(d)
v1.dot(dv)

np.float64(0.36107026789538205)

**Q3. Chunking and search by hand**

In [3]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["content"] for chunk in chunks]

batch_size = 50
X = []
# texts = [doc["content"]  for doc in documents]

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v1)
idx = np.argmax(scores)
# idx, scores[idx]
chunks[idx]["filename"]

  0%|          | 0/6 [00:00<?, ?it/s]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

**Q4. Vector search with minsearch**

In [4]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

**Q5. Text search vs vector search**

In [5]:
from minsearch import Index
from minsearch import VectorSearch

index = Index(
    text_fields=["content"],
    keyword_fields=["course"]
)

index.fit(chunks)

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

keyword_results = index.search(
    query=query,
    num_results=5
)

print("Keyword search:")
for result in keyword_results[:5]:
    print(result["filename"])

print("\nVector search:")
for result in results[:5]:
    print(result["filename"])

Keyword search:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Vector search:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


**Q6. Hybrid search**

In [10]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query = "How do I give the model access to tools?"

# Vector search
query_vector = embed.encode(query)
vector_results = vindex.search(
    query_vector,
    num_results=5
)

# Text search
text_results = index.search(
    query=query,
    num_results=5
)

# Hybrid search with Reciprocal Rank Fusion
results = rrf(
    [vector_results, text_results],
    k=60,
    num_results=5
)

results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'